In [1]:
import pandas as pd
import glob

# Load the processed train dataset
train_df = pd.read_csv('train_with_gas.csv')

# Load all_2018 to all_2023 datasets
all_files = ['all_2018_cleaned.csv', 'all_2019_cleaned.csv', 'all_2020_cleaned.csv', 'all_2021_cleaned.csv', 'all_2022_cleaned.csv', 'all_2023_cleaned.csv']

# Combine all datasets into one DataFrame
all_data = pd.concat(
    [pd.read_csv(file) for file in all_files], 
    ignore_index=True
)

# Rename columns for clarity
all_data = all_data.rename(columns={
    "Date (GMT+1)": "datetime",
    "Cross border electricity trading": "cross_border_trading",
    "Non-Renewable": "non_renewable",
    "Renewable": "renewable",
    "Load": "load",
    "CO2 Emission Allowances, Auction DE": "co2_emission_allowances"
})

# Convert the datetime column in all_data to timezone-aware datetime
all_data['datetime'] = pd.to_datetime(all_data['datetime'], format='%Y-%m-%d %H:%M:%S').dt.tz_localize('Europe/Berlin', ambiguous='infer')

# Fill missing values in all_data
numeric_columns = [
    'cross_border_trading', 'non_renewable', 'renewable', 'load', 'co2_emission_allowances'
]
for column in numeric_columns:
    all_data[column] = (
        all_data[column].interpolate(method='linear')  # Linear interpolation
        .ffill()  # Forward fill
        .bfill()  # Backward fill
    )

# Create a temporary 'datetime' column in train_df for merging
train_df['datetime'] = pd.to_datetime(train_df['ds']).dt.tz_localize('Europe/Berlin', ambiguous='infer')

# Merge all_data into train_df using the 'datetime' column
train_df = pd.merge(train_df, all_data, on='datetime', how='left')

# Drop the temporary 'datetime' column from train_df
# Keep 'ds' as-is to preserve the original nature
train_df = train_df.drop(columns=['datetime'])

# Save the updated train dataset
train_df.to_csv('train_with_all_data.csv', index=False)

print("Hourly data from all_2018 to all_2023 successfully merged into train_with_gas without altering 'ds'.")

Hourly data from all_2018 to all_2023 successfully merged into train_with_gas without altering 'ds'.
